In [ ]:
#Trial
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import pandas as pd 
from numba import njit

N = 21
Total_Population = N*N
Time = 150
Repetition = 5

#Transition_Probability = np.arange(0, 1.1, Partition)
#Apoptosis_Probability = np.arange(0, 1.1, Partition)
#ApoptoticTransition_Probability = np.arange(0.1, 1.1, Partition)
NormalCellMitosis_Probability = 0.5
CancerCellMitosis_Probability = NormalCellMitosis_Probability*2

Eff_Normal_Pob = 1-np.exp(-NormalCellMitosis_Probability)
Eff_Cancer_Prob = 1-np.exp(-CancerCellMitosis_Probability)
Eff_Apop_Trans_Prob = 1 - np.exp(-1)

Normalized_P = np.array((0, 0.5, 1, 2, 10))
Normalized_Rbar = np.array((0, 0.5, 1, 2, 10))

@njit
def apply_sir_model(center, neighbors, m, Eff_Normal_Pob, Eff_Cancer_Prob, Eff_Apop_Prob, Eff_Apop_Trans_Prob):
    if 1 in neighbors and center == 0:
        CancerCell_neighbors_count = np.sum(neighbors == 1)
        transition_probability = np.random.random()
        if transition_probability < (1-np.exp(-m*CancerCell_neighbors_count)):
            return 1
        else:
            return 0
    elif center == 1:
        apoptosis_probability = np.random.random()
        if apoptosis_probability < Eff_Apop_Prob:
            return 2
        else:
            return 1
    elif center == 2:
        apoptotictransition_probability = np.random.random()
        if apoptotictransition_probability < Eff_Apop_Trans_Prob:
            return 3
        else: 
            return 2
    elif center == 3:
        random_neighbor = neighbors[np.random.randint(0,len(neighbors))]
        if random_neighbor == 0:
            normalcellmitosis_probability = np.random.random()
            if normalcellmitosis_probability < Eff_Normal_Pob:
                return 0
            else:
                return 3
        elif random_neighbor == 1:
            cancercellmitosis_probability = np.random.random()
            if cancercellmitosis_probability < Eff_Cancer_Prob:
                return 1
            else: 
                return 3
        else:
            return 3
    else:
        return center

def Find_Neighbors(N):
    Neighbors_Array = [[[] for _ in range(N)] for _ in range(N)]  # List of lists to store neighbor coordinates

    for i in range(N):
        for j in range(N):
            neighbors = []
            
            # Possible neighbor positions
            possible_moves = [(-1, -1), (-1, 0), (-1, 1),
                              (0, -1),         (0, 1),
                              (1, -1), (1, 0), (1, 1)]
            
            for dx, dy in possible_moves:
                ni, nj = i + dx, j + dy
                if 0 <= ni < N and 0 <= nj < N:  # Check if within bounds
                    neighbors.append((ni, nj))
            
            Neighbors_Array[i][j] = neighbors  # Store the neighbors for (i, j)

    return Neighbors_Array
Neighbor_Position_Array = Find_Neighbors(N)

for m in Normalized_P:
    for l in Normalized_Rbar:
        Eff_Apop_Prob = 1 - np.exp(-l)
        for k in range(Repetition):
            repetition = k+1

            gif_save_path = rf"C:\Users\phili\OneDrive\Documents\Research\Figures\PCA_3_26_25\GIFs/Rep={repetition}P={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}.gif"
            png_save_path = rf"C:\Users\phili\OneDrive\Documents\Research\Figures\PCA_3_26_25\PNG/Population_Plot_Rep={repetition}P={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}.png"

            Initial_Array = np.zeros((N, N))
            Initial_Array[N//2, N//2] = 1
            x = list(range(0, Time + 1))

            Population_Array = np.copy(Initial_Array)
            NormalCell = [] 
            CancerCell = []
            Apoptosis = []
            Empty = []

            fig, ax = plt.subplots(figsize=(5, 5))
            cmap_custom = plt.cm.colors.ListedColormap(['darkgreen', 'maroon', 'darkblue', 'grey'])

            def update(frame):
                ax.clear()
                ax.imshow(Initial_Array, cmap=cmap_custom, aspect='auto', vmin=0, vmax=3)

                if frame == 0:
                    ax.set_title(f'Initial State, \nRep={repetition}P={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}')
                else:
                    ax.set_title(f'Iteration {frame}, \nRep={repetition}P={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}')

                NormalCell_count = 0
                CancerCell_count = 0
                Apoptosis_count = 0
                Empty_count = 0

                for i in range(N):
                    for j in range(N):

                        Location = np.array(Neighbor_Position_Array[i][j])
                        neighbors = Initial_Array[Location[:, 0], Location[:, 1]]
            
                        center = Initial_Array[i, j]
                        Population_Array[i, j] = apply_sir_model(center, neighbors, m, Eff_Normal_Pob, Eff_Cancer_Prob, Eff_Apop_Prob, Eff_Apop_Trans_Prob)

                unique, counts = np.unique(Population_Array, return_counts=True)
                cell_counts = dict(zip(unique, counts))
            
                # Get the counts (0 if not present)
                NormalCell_count = cell_counts.get(0, 0)  # Normal cells (0)
                CancerCell_count = cell_counts.get(1, 0)  # Cancer cells (1)
                Apoptosis_count = cell_counts.get(2, 0)   # Apoptotic cells (2)
                Empty_count = cell_counts.get(3, 0)       # Empty cells (3)
            
                # Store the proportions for later plotting
                NormalCell.append(NormalCell_count / Total_Population)
                CancerCell.append(CancerCell_count / Total_Population)
                Apoptosis.append(Apoptosis_count / Total_Population)
                Empty.append(Empty_count / Total_Population)
            
                # Update the grid for the next iteration
                Initial_Array[:] = Population_Array
            
            # Create the animation
            animation = FuncAnimation(fig, update, frames=Time, interval=500, repeat=False)

            # Save the animation with a different filename for each combination of m and n
            animation.save(gif_save_path, writer='imagemagick')

            # Scatter plot for susceptible, infected, and recovered
            plt.figure(figsize=(5, 5))
            plt.plot(x, NormalCell, label='Normal Cell', marker='.', color='darkgreen')
            plt.plot(x, CancerCell, label='Cancer Cell', marker='.', color='maroon')
            plt.plot(x, Apoptosis, label='Apoptotic', marker='.', color='darkblue')
            plt.plot(x, Empty, label='Empty', marker='.', color='grey')

            
            # Add labels and a legend
            plt.xlabel('Iterations')
            plt.ylabel('Population')
            plt.title(f'Rep={repetition}\nP={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}')

            plt.legend()

            # Save the scatter plot
            plt.savefig(png_save_path)
            plt.show()

            df = pd.DataFrame({
                'NormalCell': NormalCell,
                'CancerCell': CancerCell,
                'Apoptosis': Apoptosis,
                'Empty': Empty
            })
            
            # Save the DataFrame to CSV
            csv_save_path = rf"C:\Users\phili\OneDrive\Documents\Research\Raw Datas\PCA_3_26_25/Rep={repetition}P={m}_R={l}_U=1_M={NormalCellMitosis_Probability}C={CancerCellMitosis_Probability}.csv"
            df.to_csv(csv_save_path, index=False)
                
  